# Нейронные сети и обработка естественного языка - NLP

# Модуль 4. Трансформеры на практике

Библиотека `transformers` от Hugging Face — это мощный инструмент для работы с трансформерами.  
Она предоставляет готовые модели и удобные интерфейсы для обучения и генерации текста:
- `Pipelines` — конвейеры (пайплайны) для быстрого использования моделей.
- `Trainer` — для обучения своих моделей.
- `generate` — для генерации текста с помощью обученных моделей.

Помимо этого, `transformers` берет на себя все сложности, связанные с токенизацией, подготовкой данных и оптимизацией обучения.  
Вы можете сосредоточиться на творчестве и экспериментах, а не на технических деталях.

Официальная документация: https://huggingface.co/docs/transformers/index




## 1. Предварительная подготовка

### 1.1 HF-токен

**ВНИМАНИЕ**: Рекомендуется зарегистрироваться на Hugging Face и создать свой аккаунт.
Вам понадобится `HF_TOKEN` для загрузки некоторых моделей и использования API.

Как получить токен:

1. Зарегистрироваться на Hugging Face: https://huggingface.co/join
2. В меню по клику на аватар выбрать пункт [Access Tokens](https://huggingface.co/settings/tokens)

3. Нажать `New token`
4. Выбрать:

   * `read` — достаточно почти для всех курсов и скачивания моделей
5. Скопировать токен вида:

    ```text
    hf_xxxxxxxxxxxxxxxxx
    ```

Полученный токен следует разместить в файле ```__config__.py``` в виде переменной `HF_TOKEN`:

```python
HF_TOKEN = "hf_xxxxxxxxxxxxxxxxx"
```


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import torch

from __config__ import *

print(len(HF_TOKEN))


### 1.2 Хранение моделей

Трансформеры очень объемны и даже терабайтный SSD может заполниться ими довольно быстро.

Рекомендуется использовать отдельный быстрый диск для кеша моделей Hugging Face. 

Настроить кеш можно с помощью переменной окружения `HF_HOME`:

```python
import os
os.environ["HF_HOME"] = "/path/to/your/cache/directory"
```

Обычно модели сохраняются в папке `~/.cache/huggingface/`, но вы можете указать любой другой путь, который вам удобен.

Неиспользуемые модели можно безопасно удалять, чтобы освободить место.

In [ ]:
import os
os.environ["HF_TOKEN"] = HF_TOKEN
os.environ["HF_HOME"] = "./hf_cache" # для колаба ок

### 1.3 ‼️Переполнение VRAM

Что делать?

```python
import gc
import torch

for prefix in ["pipe", "model", "tokenizer"]:
    for global_name in list(globals()):
        if global_name.startswith(prefix) and global_name != 'pipeline':
            del globals()[global_name]

gc.collect()
torch.cuda.empty_cache()
torch.cuda.ipc_collect()

```

## 2. Предобученные модели и токенизаторы

`AutoTokenizer` и `AutoModel` — это универсальные классы, которые автоматически подбирают правильные токенизаторы и модели на основе указанного имени.

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-multilingual-cased")

text = "Привет, трансформеры!"
tokens = tokenizer(text, return_tensors="pt")

print(tokens)
print()
print(tokenizer.convert_ids_to_tokens(tokens["input_ids"][0]))

In [ ]:
from transformers import AutoModel

model_name = "bert-base-uncased"

model = AutoModel.from_pretrained(model_name)

## 3. Pipelines

`pipeline` — это самый простой способ использовать модель: он прячет токенизацию, запуск модели и постобработку. 

Это связка из трёх шагов: 
- preprocessing
- model inference
- postprocessing.

В HuggingFace были/существуют следующие пайплайны:
- `text-classification` — классификация текста
- `zero-shot-classification` — классификация текста без обучения на конкретных классах
- `text-generation` — генерация текста
- `token-classification` — классификация токенов (например, для NER)
- `ner` - Named Entity Recognition (распознавание именованных сущностей)
- `fill-mask` — заполнение пропусков в тексте
- `feature-extraction` — извлечение признаков из текста (эмбеддинги)

Нижеперечисленные пайплайны были заменены на `text-generation`, так как современные модели, такие как GPT-3.5 и GPT-4, способны выполнять все эти задачи в рамках генерации текста, но их до сих пор можно использовать с более ранними версиями `transformers` (до версии 5):
- `question-answering` — ответ на вопрос по тексту (заменен на `text-generation`)
- `summarization` — суммаризация текста (заменен на `text-generation`)
- `translation` — перевод текста (заменен на `text-generation`)
- `text2text-generation` — генерация текста на основе другого текста (заменен на `text-generation`)
- `conversational` — для создания чат-ботов и диалоговых систем (заменен на `text-generation`)
- `text-to-code` — генерация кода на основе текста (заменен на `text-generation`)
- `text-to-table` — генерация таблиц на основе текста (заменен на `text-generation`)
- `text-to-text` — генерация текста на основе другого текста  (заменен на `text-generation`)


In [ ]:
from transformers import pipeline

### 3.1 Классификация текста

In [ ]:
pipe_classifier = pipeline("sentiment-analysis")

pipe_classifier([
    "I love this course!",
    "This library is confusing."
])

In [ ]:
# для русского языка
pipe_classifier_ru = pipeline(
    "sentiment-analysis",
    model="cointegrated/rubert-tiny-sentiment-balanced"
)

pipe_classifier_ru("Этот курс получился очень полезным.")

### 3.2 Zero-shot classification

Классификация без дообучения: даём текст и список возможных классов.

In [ ]:
pipe_zero_shot = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli"
)

In [ ]:
pipe_zero_shot(
    "The model extracts named entities from text.",
    candidate_labels=["NLP", "computer vision", "databases", "network security"]
)

### 3.3  Генерация текста

In [ ]:
pipe_generator = pipeline(
    "text-generation",
    model="gpt2"
)

In [ ]:
from IPython.display import HTML, display

output = pipe_generator(
    "In machine learning, transfer learning means",
    # max_new_tokens=100,
    max_length=100,
    do_sample=True,
    temperature=0.8
)

generated_text = output[0]["generated_text"]

display(HTML(generated_text.replace("\n", "<br>")))

### 3.4 Fill-mask

Задача для BERT-подобных моделей: угадать пропущенное слово.

In [ ]:
pipe_fill_mask = pipeline(
    "fill-mask",
    model="bert-base-multilingual-cased"
)

In [ ]:
pipe_fill_mask("Paris is the capital of [MASK].")

### 3.5 Named Entity Recognition, NER

Извлечение сущностей: имена, организации, география.

In [ ]:
pipe_ner = pipeline(
    "token-classification",
    model="Davlan/bert-base-multilingual-cased-ner-hrl",
    aggregation_strategy="simple"
)

pipe_ner("Илон Маск основал SpaceX в США.")

In [ ]:
# для русского языка

pipe_ner_ru = pipeline(
    "ner", 
    model="Gherman/bert-base-NER-Russian", 
    aggregation_strategy="simple"
)

In [ ]:
# Обработка текста
text = """
Компания "Яндекс" была основана Аркадием Воложем и Ильей Сегаловичем в России в 1993 году."
"""
entities = pipe_ner_ru(text)

for entity in entities:
    print(f"Сущность: {entity['word']}, Тип: {entity['entity_group']}")

In [ ]:
pipe_ner_address = pipeline(
    "ner", 
    model="aidarmusin/address-ner-ru", 
    aggregation_strategy="simple"
)

In [ ]:
text = """
Мой адрес: Россия, Москва, ул. Тверская, д. 1.
Твой адрес: Россия, Санкт-Петербург, Невский проспект, д. 10.
Их адрес: Россия, Новосибирск, ул. Ленина, д. 5.
Their address: 254 Elm Street Seattle WA 44356 USA
"""

entities = pipe_ner_address(text)

for entity in entities:
    print(f"Сущность: {entity['word']}, \tТип: {entity['entity_group']}")

### 3.6 Feature Extraction

Получение эмбеддингов для текста, которые можно использовать в других задачах. Например, для кластеризации, поиска по семантике или в качестве признаков для других моделей.

In [ ]:
texts = [
    "I sat near the river bank.",
    "The bank approved my loan.",
    "Children played on the river bank.",
    "The central bank increased rates.",
    "The bank angle of the plane has been rather steep on approach."
]

In [ ]:
pipe_extractor = pipeline(
    "feature-extraction",
    model="sentence-transformers/all-MiniLM-L6-v2"
)

emb = pipe_extractor(texts)

for text, emb_vector in zip(texts, emb):
    print(text)
    print()
    print(np.array(emb_vector).shape)



In [ ]:
bank_embeddings = []

for text in texts:
    # pipeline вернет [batch, tokens, hidden_size]
    features = pipe_extractor(text, return_tensors=False)[0]

    tokens = tokenizer.tokenize(text)
    tokens = [tokenizer.cls_token] + tokens + [tokenizer.sep_token]

    bank_idx = tokens.index("bank")

    bank_embeddings.append({
        "text": text,
        "embedding": np.array(features[bank_idx])
    })

    print(text)
    print(tokens)
    print("bank index:", bank_idx)
    print()

In [ ]:
from sklearn.decomposition import PCA

X = np.array([item["embedding"] for item in bank_embeddings])

pca = PCA(n_components=2)
X_2d = pca.fit_transform(X)

In [ ]:
plt.figure(figsize=(10, 8))

for i, item in enumerate(bank_embeddings):
    x, y = X_2d[i]

    plt.scatter(x, y, s=120)
    plt.text(
        x + 0.02,
        y + 0.02,
        item["text"],
        fontsize=10
    )

plt.title("Contextual embeddings of the token 'bank'")
plt.xlabel("PCA 1")
plt.ylabel("PCA 2")
plt.grid(True)
plt.show()

**ПРАКТИКА**


Используя `pipeline`, извлеките все наименовая городов из текстов новостей [Lenta.ru (short)](https://huggingface.co/datasets/zloelias/lenta-ru-short).

In [ ]:
# ваш код здесь





### Отсутствующие пайплайны

- `question-answering` — ответ на вопрос по тексту. 
- `summarization` — суммаризация текста.
- `translation` — перевод текста.

Как решать эти задачи, если нужного пайплайна нет?

Можно использовать модели напрямую через `transformers` и реализовывать необходимую логику самостоятельно.

Можно использовать LLM и решить эти задачи через инструкционные модели, задавая им правильные промпты.

## 4. Промпт-инженерия и пайплайны

Начиная с версии 5 в `transformers` предлагается для вышеупомянутых задач использовать LLM и промпт-инженерию вместо специализированных моделей.

In [ ]:
from transformers import pipeline

from transformers import GenerationConfig

gen_config = GenerationConfig(
    max_new_tokens=100,
    do_sample=False,
)

pipe_instr = pipeline(
    "text-generation",
    model="Qwen/Qwen2.5-0.5B-Instruct",
    device_map="auto",
)

### 4.1 Question Answering



In [ ]:
context = """
Лицензия GPL разрешает копировать, распространять и изменять программу
при соблюдении условий лицензии.
"""

question = "Что разрешает лицензия GPL?"

prompt = f"""
Ответь на вопрос только по контексту.

Контекст:
{context}

Вопрос:
{question}

Ответ:
"""

result = pipe_instr(
    prompt,
    do_sample=False,
    generation_config=gen_config
)

print(result[0]["generated_text"])

In [ ]:
context = """
Лицензия GPL разрешает копировать, распространять и изменять программу
при соблюдении условий лицензии.
"""

question = "Что разрешает лицензия GPL?"

messages = [
    {
        "role": "system",
        "content": "Отвечай только по предоставленному контексту."
    },
    {
        "role": "user",
        "content": f"""
Контекст:
{context}

Вопрос:
{question}
"""
    }
]

pipe_instr.model.generation_config.max_new_tokens = 100
pipe_instr.model.generation_config.max_length = None

result = pipe_instr(
    messages,
    do_sample=False,
    generation_config=gen_config
)

print(result[0]["generated_text"][-1]["content"])

### 4.2 Суммаризация текста

In [ ]:

text = """
Transformers is an open-source library developed by Hugging Face.
It provides thousands of pretrained machine learning models for
natural language processing, computer vision, audio processing,
and multimodal tasks.

The library supports tasks such as text classification,
question answering, summarization, translation, text generation,
named entity recognition, and image classification.

Transformers integrates with PyTorch, TensorFlow, and JAX,
making it widely used in research and production systems.
"""

messages = [
    {
        "role": "system",
        "content": """
You are a professional summarization system.

Rules:
- summarize only
- do not repeat the original text
- do not quote
- keep only key facts
- maximum 2 sentences
- concise technical style
"""
    },
    {
        "role": "user",
        "content": f"""
Summarize the following text.

TEXT:
{text}

SUMMARY:
"""
    }
]

result = pipe_instr(
    messages,
    do_sample=False, # не использовать сэмплирование, а брать наиболее вероятный ответ
    return_full_text=False, # не возвращать исходный текст в ответе
    repetition_penalty=1.2, # штрафовать за повторение слов
    max_new_tokens=50, # ограничить длину ответа
    generation_config=gen_config
)

print(text)
print()

HTML(result[0]["generated_text"])

### 4.3 Машинный перевод

In [ ]:
text = """
Transformers is an open-source library developed by Hugging Face.
It provides pretrained models for natural language processing,
computer vision, audio processing, and multimodal AI systems.
"""

In [ ]:
messages = [
    {
        "role": "system",
        "content": """
You are a professional translator.

Rules:
- translate accurately
- preserve meaning
- preserve technical terms
- do not explain
- do not summarize
- return translation only
"""
    },
    {
        "role": "user",
        "content": f"""
Translate the following text from English to Russian.

TEXT:
{text}

TRANSLATION:
"""
    }
]

result = pipe_instr(
    messages,
    generation_config=gen_config,
    return_full_text=False
)

print(text)
print()

HTML(result[0]["generated_text"])

**ПРАКТИКА** 

Используя `pipeline`, переведите текст рандомных 20 новостных статей с русского на английский язык из датасета [Lenta.ru (short)](https://huggingface.co/datasets/zloelias/lenta-ru-short).



In [ ]:
# ваш код здесь




## 5. generate



In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_name = "sberbank-ai/rugpt3small_based_on_gpt2"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

prompt = "Жили-были дед и"

inputs = tokenizer(prompt, return_tensors="pt")

outputs = model.generate(
    **inputs,
    max_new_tokens=30
)

text = tokenizer.decode(outputs[0], skip_special_tokens=True)

print(text)

Обратим внимание на параметры настройки `generate`:
- `do_sample` — включает режим сэмплирования, который позволяет генерировать более разнообразные ответы. Если `False`, используется жадный поиск (greedy decoding).
- `max_new_tokens` — максимальное количество новых токенов, которые модель может сгенерировать. Это ограничивает длину ответа.
- `temperature` — параметр, который контролирует степень случайности в генерации. Чем
- `top_k` — ограничивает выбор следующего токена только топ-k наиболее вероятными вариантами.
- `top_p` — ограничивает выбор следующего токена только теми, которые вместе составляют топ-p вероятности (nucleus sampling).
- `num_beams` — количество лучей для поиска (beam search), который позволяет генерировать более качественные и разнообразные ответы, но увеличивает время генерации.

In [ ]:
prompt = "Без труда не вытащишь и рыбку"

inputs = tokenizer(prompt, return_tensors="pt")

outputs = model.generate(
    **inputs,
    # do_sample=True,
    # temperature=0.2,
    # top_k=50,
    # top_p=0.95,
    # num_beams=5,
    max_new_tokens=30,
)

print( tokenizer.decode(outputs[0], skip_special_tokens=True) )

### Пишем свой ChatGPT

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "Qwen/Qwen2.5-0.5B-Instruct"  # маленькая instruct-модель

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)

model.eval()

In [ ]:
# используем функцию apply_chat_template для создания промпта в стиле диалога
user_text = "Каков ответ на вопрос жизни, смерти, вселенной и всего остального?"

messages = [ 
    {
        "role": "system",
        "content": "Ты полезный русскоязычный ассистент. Отвечай кратко и понятно."
    },
    {
        "role": "user",
        "content": user_text
    }]

input_ids = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True, # добавляет специальный токен в конце, чтобы модель понимала, что нужно генерировать ответ
    tokenize=True, # сразу токенизировать и возвращать input_ids
    return_dict=True, # возвращать словарь с input_ids и attention_mask
    return_tensors="pt"
)

print(input_ids)

print()

print(tokenizer.batch_decode(input_ids["input_ids"], skip_special_tokens=True)[0])

In [ ]:
messages = [
    {
        "role": "system",
        "content": "Ты полезный русскоязычный ассистент. Отвечай кратко и понятно."
    }
]

while True:
    user_text = input("Вы: ")

    if user_text.lower() in ["exit", "quit", "выход", "стоп"]:
        print("Диалог завершён.")
        break

    messages.append({
        "role": "user",
        "content": user_text
    })

    input_ids = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            input_ids['input_ids'],
            max_new_tokens=300,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.1,
            eos_token_id=tokenizer.eos_token_id
        )

    new_tokens = output_ids[0][input_ids['input_ids'].shape[-1]:]
    answer = tokenizer.decode(new_tokens, skip_special_tokens=True)

    print("Бот:", answer)

    messages.append({
        "role": "assistant",
        "content": answer
    })

**ПРАКТИКА**

Поэкспериментируйте с параметрами `generate` и промптами, чтобы создать свою версию ChatGPT, которая может отвечать на вопросы, вести диалог или выполнять другие задачи.


In [ ]:
# ваш код здесь



